# Logistic Regression & Classification Threshold Tuning
Accompanies **Chapter 4: Loss Functions and Metrics** and **Chapter 9: Logistic Regression and Binary Classification**.

In this notebook, we:
1. Fit a `LogisticRegression` model on the Bank Deposit subscription dataset (`bd_train.csv`).
2. Calculate predicted probabilities $\hat{p}_i = \sigma(\mathbf{w}^T \mathbf{x}_i)$.
3. Compute and plot the **Kolmogorov-Smirnov (KS) Statistic**.
4. Plot the **ROC Curve** and calculate the **AUC**.
5. Optimize the decision cutoff $c^*$ for maximum $F_1$ score.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_curve, roc_auc_score, confusion_matrix, classification_report

data_path = os.path.abspath('../../data/bd_train.csv')
df = pd.read_csv(data_path)

numeric_cols = ['age', 'balance', 'day', 'duration', 'campaign', 'pdays', 'previous']
X = df[numeric_cols].fillna(df[numeric_cols].median())
y = (df['y'] == 'yes').astype(int)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

model = LogisticRegression(max_iter=1000, random_state=42).fit(X_train_scaled, y_train)
y_probs = model.predict_proba(X_test_scaled)[:, 1]

fpr, tpr, thresholds = roc_curve(y_test, y_probs)
ks_values = tpr - fpr
ks_max = np.max(ks_values)
ks_cutoff = thresholds[np.argmax(ks_values)]

plt.figure(figsize=(8, 5))
plt.plot(thresholds, tpr, label='True Positive Rate (Responders CDF)', color='green', linewidth=2)
plt.plot(thresholds, fpr, label='False Positive Rate (Non-Responders CDF)', color='red', linewidth=2)
plt.plot(thresholds, ks_values, label=f'KS Curve (Max KS = {ks_max:.3f} at c = {ks_cutoff:.3f})', color='blue', linestyle='--', linewidth=2)
plt.xlim([0, 1])
plt.ylim([0, 1])
plt.xlabel('Probability Cutoff Threshold ($c$)', fontsize=11)
plt.ylabel('Cumulative Rate', fontsize=11)
plt.title('Kolmogorov-Smirnov (KS) Separation Analysis', fontsize=12, fontweight='bold')
plt.legend(loc='lower center', fontsize=10)
plt.grid(True, alpha=0.3)
plt.show()

y_pred_opt = (y_probs >= ks_cutoff).astype(int)
print(classification_report(y_test, y_pred_opt, target_names=['No', 'Yes']))